# Capacity Test Analysis Template
<img src="./images/vdelogo.jpg" width="100" height="50">

**Project Name:** [Enter your project name here]  
**Test Date:** [Enter test date]  
**Analyst:** [Enter analyst name]

This notebook provides a template for conducting capacity tests using pvcaptest following ASTM E2848 or IEC 61724-2 standards.

## Introduction

This template guides you through a complete capacity test workflow. **You must customize all parameters marked with placeholders** (e.g., `[YOUR_VALUE]`) for your specific project.

The workflow includes:
1. Standard selection (ASTM vs IEC)
2. Module type configuration
3. Data loading and column grouping
4. Regression column mapping
5. Sensor aggregation
6. Data filtering (multiple filter types)
7. Reporting conditions calculation
8. Regression analysis
9. Capacity test results

**Important:** Refer to `docs/user_guide/configurable_parameters.md` for detailed explanations of all parameters.

## Imports

Import required packages for the analysis.

In [1]:
import warnings
# warnings.filterwarnings('ignore')  # Uncomment to suppress warnings in final report
import pandas as pd
import captest as ct

# Display version information
print(f"pvcaptest version: {ct.__version__}")

# Set pandas display options
pd.set_option('max_colwidth', 120)

pvcaptest version: 0.0.post1.dev1017+g63597f249


## Global Test Parameters

Define project-specific parameters that will be used throughout the analysis. **Update all values below for your project.**

In [2]:
# Global test parameters - UPDATE THESE FOR YOUR PROJECT
TOLERANCE = '+/- 7'        # Capacity test tolerance (e.g., '+/- 7', '- 5')
AC_NAMEPLATE = 1_000_000   # AC nameplate capacity in Watts (e.g., 1 MW = 1,000,000 W)
HRS_REQ = 12.5            # Required hours of data (ASTM E2848 default: 12.5 hours)
MIN_IRR = 200             # Minimum irradiance filter (W/m²) - typical: 200-400
MAX_IRR = 1200            # Maximum irradiance filter (W/m²) - typical: 1200-1400
SIM_DAYS = 60             # Number of days for simulation period (if applicable)

# Bifacial parameters (set to None for monofacial systems)
BIFACIALITY = None         # Bifaciality factor (0.0-1.0, typically 0.7-0.9) or None for monofacial

# Reporting conditions filter parameters
rep_irr_filter = 0.20      # Percent band around reporting irradiance (default: 0.20 = ±20%)
IRR_LOW = 1 - rep_irr_filter   # Lower bound (e.g., 0.8 = 80% of reporting irradiance)
IRR_HIGH = rep_irr_filter + 1  # Upper bound (e.g., 1.2 = 120% of reporting irradiance)

## 1. Standard Selection

**Parameter:** `standard` - Choose between ASTM E2848 or IEC 61724-2

- **ASTM E2848** (default): Standard Test Method for Reporting Photovoltaic Non-Concentrator System Performance
- **IEC 61724-2**: Photovoltaic system performance monitoring - Capacity evaluation method

**Impact:** The standard selection affects:
- Regression formula (ASTM uses 4-term, IEC uses 2-term)
- Required regression columns
- Reporting conditions calculation method

**Reference:** See `docs/user_guide/configurable_parameters.md` section 1 for details.

In [3]:
# Select standard: 'ASTM' or 'IEC'
# Default is 'ASTM' if not specified
STANDARD = 'ASTM'  # Change to 'IEC' if using IEC 61724-2

print(f"Selected standard: {STANDARD}")

Selected standard: ASTM


## 2. Module Type Configuration

### 2.1 Bifacial vs Monofacial

**For Bifacial Modules:**
- Calculate total irradiance: $E_{TOTAL} = E_{POA} + E_{REAR} \times \varphi$
- Update regression columns to use $E_{TOTAL}$ instead of $E_{POA}$

**For Monofacial Modules:**
- Use standard POA irradiance (no changes needed)

**Reference:** See `docs/user_guide/configurable_parameters.md` section 2.1 and `docs/user_guide/bifacial.rst` for details.

In [4]:
# Bifacial configuration (skip this cell if using monofacial modules)
# Uncomment and customize if your project uses bifacial modules

# if BIFACIALITY is not None:
#     # Calculate E_Total after loading data (see Data Loading section)
#     # meas.data['irr_poa_tot'] = meas.loc['irr_poa'].mean(axis=1) + meas.loc['irr_rpoa'].mean(axis=1) * BIFACIALITY
#     # meas.reset_filter()  # Update data_filtered with new column
#     pass

### 2.2 IEC Module Type Parameters

**For IEC Standard Only:** Set module type and racking configuration to calculate cell temperature.

**Options:**
- `module_type`: `'glass_cell_poly'` (default), `'glass_cell_glass'`, `'poly_tf_steel'`
- `racking`: `'open_rack'` (default), `'close_roof_mount'`, `'insulated_back'`

**Reference:** See `docs/user_guide/configurable_parameters.md` section 2.2 for details.

In [5]:
# IEC module type parameters (only needed if STANDARD == 'IEC')
# Uncomment and customize if using IEC standard

# meas.set_iec_params(
#     module_type='glass_cell_poly',  # Options: 'glass_cell_poly', 'glass_cell_glass', 'poly_tf_steel'
#     racking='open_rack'              # Options: 'open_rack', 'close_roof_mount', 'insulated_back'
# )

## 3. Data Loading

**Parameters:** `path`, `name`, `group_columns`, `standard`, `site`

Load measured data from SCADA/DAS system. Site information enables clear sky modeling for filtering.

**Reference:** See `docs/user_guide/configurable_parameters.md` section 3 for details.

In [6]:
# Load site configuration from YAML file or define as dictionary
# Option 1: Load from site.yml file
site = './site.yml'  # Path to site.yml file

# Option 2: Define site dictionary directly (uncomment to use)
# site = {
#     'loc': {
#         'latitude': 40.0,      # Update with your site latitude
#         'longitude': -105.0,   # Update with your site longitude
#         'altitude': 1600.0,    # Update with your site elevation (meters)
#         'tz': 'America/Denver' # Update with your timezone (IANA timezone name)
#     },
#     'sys': {
#         'surface_tilt': 20,        # Module tilt angle (degrees from horizontal)
#         'surface_azimuth': 180,   # Module azimuth (0=North, 90=East, 180=South, 270=West)
#         'albedo': 0.2             # Ground albedo (reflectivity)
#     }
# }

In [7]:
# Load measured data
# UPDATE THE PATH TO YOUR DATA FILE
meas = ct.load_data(
    './data/example_measured_data.csv',  # UPDATE: Path to your measured data file
    name='meas',                          # Identifier for this CapData object
    standard=STANDARD,                    # Use the standard selected above
    site=site,                            # Site configuration for clear sky modeling
    group_columns='./column_groups.xlsx', # Path to column groups file (or None for auto-grouping)
    # column_groups_template=True,        # Uncomment to generate template Excel file
    reindex=True,                         # Reindex to ensure proper datetime index
    verbose=True                          # Print information about loading process
)

# Display first few rows to verify data loaded correctly
meas.data.head(3)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 5, saw 2


## 4. Column Grouping

**Parameter:** `column_groups` - Maps column names to measurement types

pvcaptest automatically groups columns by measurement type. Review the automatic grouping and create a custom `column_groups.xlsx` file if needed.

**Workflow:**
1. Review automatic grouping (below)
2. If incorrect, generate template: set `column_groups_template=True` in `load_data()`
3. Edit the generated `column_groups.xlsx` file
4. Reload data with `group_columns='./column_groups.xlsx'`

**Reference:** See `docs/user_guide/configurable_parameters.md` section 4 for details.

In [ ]:
# Review automatic column grouping
# Check if columns are correctly grouped
print("Column groups:")
meas.column_groups

# Display specific groups (update group names based on your data)
# Example: meas.column_groups.irr_poa  # POA irradiance columns
# Example: meas.column_groups.real_pwr_mtr  # Power meter columns

### Generate Column Groups Template (if needed)

If automatic grouping is incorrect, uncomment the cell below to generate a template Excel file for manual editing.

In [ ]:
# Uncomment to generate column_groups.xlsx template
# This will create a file with all column headers that you can edit in Excel
# meas_template = ct.load_data(
#     './data/example_measured_data.csv',
#     column_groups_template=True  # Generates column_groups.xlsx in data directory
# )
# After editing, reload data with: group_columns='./column_groups.xlsx'

## 5. Bifacial Total Irradiance Calculation (if applicable)

**For bifacial systems only:** Calculate $E_{TOTAL} = E_{POA} + E_{REAR} \times \varphi$

Skip this section if using monofacial modules.

In [ ]:
# Bifacial total irradiance calculation
# Uncomment and customize if using bifacial modules

# if BIFACIALITY is not None:
#     # Calculate E_Total using column group names (update based on your column groups)
#     # meas.data['irr_poa_tot'] = meas.loc['irr_poa'].mean(axis=1) + meas.loc['irr_rpoa'].mean(axis=1) * BIFACIALITY
#     
#     # Update data_filtered to include the new column
#     # meas.reset_filter()
#     
#     print(f"Bifaciality factor: {BIFACIALITY}")
#     print("E_Total calculated and added to data")
# else:
#     print("Monofacial system - skipping E_Total calculation")

## 6. Regression Column Mapping

**Parameter:** `regression_cols` - Maps regression variables to data columns

**Required mappings:**
- **ASTM:** `power`, `poa`, `t_amb`, `w_vel`
- **IEC:** `power`, `poa`

**How to set:** Use column group keys (from `column_groups`) or direct column names.

**Reference:** See `docs/user_guide/configurable_parameters.md` section 5 for details.

In [ ]:
# Set regression columns
# UPDATE the column group names or column names to match your data
# Option 1: Using column group keys (recommended)
meas.set_regression_cols(
    power='real_pwr_mtr',    # UPDATE: Power column/group (e.g., 'real_pwr_mtr', 'meter_power')
    poa='irr_poa',           # UPDATE: POA irradiance column/group (or 'irr_poa_tot' for bifacial)
    t_amb='temp_amb',        # UPDATE: Ambient temperature column/group (ASTM only)
    w_vel='wind_speed'       # UPDATE: Wind speed column/group (ASTM only)
)

# Option 2: Using direct column names (if not using column groups)
# meas.set_regression_cols(
#     power='Meter_Power',
#     poa='POA_Irradiance',
#     t_amb='Ambient_Temp',
#     w_vel='Wind_Speed'
# )

# Verify regression columns mapping
print("Regression columns mapping:")
meas.regression_cols

## 7. Sensor Aggregation

**Parameter:** `agg_map` in `agg_sensors()` - Aggregates multiple sensors of the same type

**Default behavior:** Sums power, averages POA/temperature/wind speed

**When to use:** When multiple sensors exist and you want to combine them into single values.

**Reference:** See `docs/user_guide/configurable_parameters.md` section 6 for details.

In [ ]:
# Aggregate sensors (if multiple sensors of the same type exist)
# Option 1: Use defaults (sum power, mean others)
# meas.agg_sensors()

# Option 2: Custom aggregation (uncomment and customize)
# meas.agg_sensors(agg_map={
#     'real_pwr_inv': 'sum',      # Sum inverter power
#     'irr_poa': 'mean',          # Average POA irradiance
#     'temp_amb': 'mean',         # Average ambient temperature
#     'wind_speed': 'mean'        # Average wind speed
# })

# If aggregation was used, regression columns are automatically updated
# Check updated regression columns:
# meas.regression_cols

### Sensor Outliers

Check whether sensors are installed and aligning properly

## 8. Data Filtering

Apply filtering steps to remove data points that don't meet test criteria. Filtering order can affect results - apply filters in logical sequence.

**Reference:** See `docs/user_guide/configurable_parameters.md` section 7 for detailed filter parameter explanations.

In [ ]:
# Reset filter to start fresh (copies data to data_filtered)
meas.reset_filter()

### 8.1 Initial Irradiance Filter

**Parameter:** `filter_irr(low, high, ref_val, col_name)`

Remove data outside acceptable irradiance range. Typically applied early in filtering sequence.

**Typical values:** Initial filter: 200-1200 W/m²

In [ ]:
# Initial irradiance filter
# Remove data below MIN_IRR and above MAX_IRR (absolute values in W/m²)
meas.filter_irr(MIN_IRR, MAX_IRR)

# Alternative: Filter as percentage of reference value
# meas.filter_irr(0.8, 1.2, ref_val=800)  # 80% to 120% of 800 W/m²

# View filtering summary
meas.get_summary()

### 8.2 Sensor Consistency Filter

**Parameter:** `filter_sensors(perc_diff)` - Removes periods when sensors disagree

**Default:** 5% difference threshold for POA sensors

**When to use:** When multiple sensors of the same type exist and should agree within tolerance.

In [ ]:
# Sensor consistency filter
# Default: 5% difference for POA sensors
meas.filter_sensors()

# Custom thresholds (uncomment to use)
# meas.filter_sensors(perc_diff={
#     'irr_poa': 0.05,      # 5% difference for POA irradiance
#     'temp_amb': 0.10      # 10% difference for ambient temperature
# })

meas.get_summary()

### 8.3 Outlier Filter

**Parameter:** `filter_outliers(contamination, support_fraction)` - Removes statistical outliers

**Default:** 4% contamination (removes 4% of points as outliers)

**When to use:** To remove obvious outliers visible in scatter plots.

In [ ]:
# Outlier filter using elliptic envelope
# Default: 4% contamination
meas.filter_outliers()

# Custom contamination (uncomment to use)
# meas.filter_outliers(contamination=0.05)  # 5% outliers

meas.get_summary()

### 8.4 Power Filter

**Parameter:** `filter_power(power, percent, columns)` - Removes clipping or over-power conditions

**When to use:** To remove data when system is at or near maximum power (clipping).

In [ ]:
# Power filter (optional - uncomment if needed)
# Remove data at or above specified power threshold

# Option 1: Absolute threshold
# meas.filter_power(6000000)  # Remove data >= 6 MW

# Option 2: Percentage of nameplate
# meas.filter_power(AC_NAMEPLATE, percent=0.99)  # Remove data >= 99% of nameplate

# meas.get_summary()

### 8.5 Time Filter

**Parameter:** `filter_time(start, end, days, test_date, wrap_year)` - Selects specific time period

**When to use:** To focus analysis on specific test period.

In [ ]:
# Time filter (optional - uncomment if needed)
# Select specific time period for analysis

# Option 1: Date range
# meas.filter_time(start='2023-06-01', end='2023-08-31')

# Option 2: Days from start date
# meas.filter_time(start='2023-06-01', days=60)

# Option 3: Centered on test date
# meas.filter_time(test_date='2023-07-15', days=60)

# meas.get_summary()

### 8.6 Clear Sky Filter

**Parameter:** `filter_clearsky(window_length, ghi_col, keep_clear)` - Filters based on clear sky conditions

**Default:** Keeps clear periods, 20-minute window for 5-min data

**When to use:** To focus on clear sky periods (recommended for capacity tests).

In [ ]:
# Clear sky filter
# Default: Keeps clear periods, auto-detects GHI column
meas.filter_clearsky()

# Custom options (uncomment to use)
# meas.filter_clearsky(
#     window_length=20,        # Sliding window in minutes (default: 20 for 5-min data, 10 for 1-min)
#     ghi_col='irr_ghi',       # GHI column name (auto-detected if None)
#     keep_clear=True          # True = keep clear periods, False = keep cloudy periods
# )

meas.get_summary()

### 8.7 Shade Filter

**Parameter:** `filter_shade(fshdbm, query_str)` - Removes periods with shading

**When to use:** For PVsyst data or when shading data is available.

In [ ]:
# Shade filter (optional - typically for PVsyst data)
# Default: Removes all shading (fshdbm < 1.0)

# meas.filter_shade()

# Custom options (uncomment to use)
# meas.filter_shade(fshdbm=0.95)  # Remove if shading > 5%
# meas.filter_shade(query_str='ShdLoss<=50')  # Custom query string

# meas.get_summary()

### 8.8 Power Factor Filter

**Parameter:** `filter_pf(pf)` - Removes periods with low power factor

**Default:** Keeps only data with PF >= 0.999

**When to use:** When power factor data is available and quality issues exist.

In [ ]:
# Power factor filter (optional - uncomment if power factor data available)
# meas.filter_pf(0.999)  # Keep only data with PF >= 0.999

# meas.get_summary()

### 8.9 Missing Data Filter

**Parameter:** `filter_missing(columns)` - Removes periods with missing critical data

**Default:** Checks regression columns

**When to use:** To ensure all required data is present for regression.

In [ ]:
# Missing data filter
# Default: Checks regression columns
meas.filter_missing()

# Specific columns (uncomment to use)
# meas.filter_missing(columns=['power', 'poa', 't_amb', 'w_vel'])

meas.get_summary()

## 9. Reporting Conditions

**Parameter:** `rep_cond(irr_bal, percent_filter, func, freq, w_vel, rc_kwargs)` - Calculates reporting conditions

**Recommended:** Use balanced irradiance method (`irr_bal=True`) with 20% filter

**Reference:** See `docs/user_guide/configurable_parameters.md` section 8 for detailed parameter explanations.

In [ ]:
# Calculate reporting conditions
# Recommended: Use balanced irradiance method
meas.rep_cond(
    irr_bal=True,              # Use balanced irradiance method (recommended)
    percent_filter=20,          # Percent band around reporting irradiance (default: 20%)
    # freq='MS',                # Optional: Group by frequency (e.g., 'MS' = monthly)
    # func={'poa': 'median', 't_amb': 'mean', 'w_vel': 'mean'},  # Custom aggregation
    # w_vel=2.0,                # Optional: Override wind speed reporting condition
    # rc_kwargs={               # Advanced ReportingIrradiance parameters
    #     'min_percent_below': 40,
    #     'max_percent_above': 60,
    #     'min_ref_irradiance': 600,
    #     'max_ref_irradiance': 900,
    #     'points_required': 750
    # }
)

# Display reporting conditions
print("Reporting conditions:")
meas.rc

### Final Irradiance Filter Around Reporting Conditions

Filter data to ±20% (or custom percentage) around the reporting irradiance for final regression.

In [ ]:
# Final irradiance filter around reporting conditions
# Filter to ±20% (or custom percentage) of reporting irradiance
meas.filter_irr(IRR_LOW, IRR_HIGH, ref_val=meas.rc['poa'][0])

print(f"Filtering to {IRR_LOW:.1%} to {IRR_HIGH:.1%} of reporting irradiance ({meas.rc['poa'][0]:.1f} W/m²)")

meas.get_summary()

## 10. Regression Analysis

**Parameter:** `fit_regression(filter, inplace, summary)` - Fits regression model to filtered data

**Options:**
- `filter=True`: Removes outliers using residuals (> 2 std dev)
- `summary=True/False`: Print regression summary

**Reference:** See `docs/user_guide/configurable_parameters.md` section 10 for details.

In [ ]:
# Preliminary regression with residual filtering (ASTM E2848 9.1.3)
# Removes points where residual > 2 standard deviations
meas.fit_regression(filter=True, summary=True)

# View regression results
print("\nRegression coefficients:")
print(meas.regression_results.params)
print("\nP-values:")
print(meas.regression_results.pvalues)

### Final Regression Fit

After residual filtering, fit final regression without additional filtering.

In [ ]:
# Final regression fit (no additional filtering)
meas.fit_regression(filter=False, summary=True)

# Store regression results for later use
meas_regression_results = meas.regression_results

## 11. IEC-Specific Parameters (if using IEC standard)

**Parameter:** `set_iec_params(beta, delta_t, e_ref, t_stc, module_type, racking)` - Sets IEC temperature correction parameters

**Required for IEC:** `beta` (temperature coefficient) must be set from module datasheet

**Reference:** See `docs/user_guide/configurable_parameters.md` section 9 for details.

In [ ]:
# IEC parameters (only needed if STANDARD == 'IEC')
# Uncomment and customize if using IEC standard

# meas.set_iec_params(
#     beta=-0.35,              # REQUIRED: Temperature coefficient (%/°C) from module datasheet
#     delta_t=3.0,             # Temperature difference constant (°C, default: 3.0)
#     e_ref=1000.0,            # Reference irradiance (W/m², default: 1000.0)
#     t_stc=25.0,              # STC temperature (°C, default: 25.0)
#     module_type='glass_cell_poly',  # Module type (see section 2.2)
#     racking='open_rack'              # Racking type (see section 2.2)
# )

## 12. PVsyst Data Loading and Processing

Load and process PVsyst simulation data using the same workflow as measured data.

In [ ]:
# Load PVsyst data
# UPDATE THE PATH TO YOUR PVSYST OUTPUT FILE
sim = ct.load_pvsyst(
    './pvsyst/pvsyst_output_template.csv',  # UPDATE: Path to your PVsyst output file
    standard=STANDARD                       # Use the same standard as measured data
)

# Display first few rows
sim.data.head(3)

### PVsyst Bifacial Total Irradiance (if applicable)

For bifacial systems, calculate total irradiance from PVsyst output.

In [ ]:
# Bifacial total irradiance for PVsyst data (if applicable)
# Uncomment and customize if using bifacial modules

# if BIFACIALITY is not None:
#     # Calculate rear POA (if not already in data)
#     # sim.data['irr_poa_back'] = sim.data['GlobBak'] + sim.data['BackShd']
#     
#     # Calculate E_Total
#     # sim.data['irr_poa_tot'] = sim.data['GlobInc'] + sim.data['irr_poa_back'] * BIFACIALITY
#     pass

### PVsyst Regression Columns

Set regression columns for PVsyst data (typically different column names than measured data).

In [ ]:
# Set regression columns for PVsyst data
# UPDATE column names to match your PVsyst output
sim.set_regression_cols(
    power='E_Grid',          # UPDATE: Power column name in PVsyst output
    poa='GlobInc',           # UPDATE: POA irradiance (or 'irr_poa_tot' for bifacial)
    t_amb='T_Amb',           # UPDATE: Ambient temperature (ASTM only)
    w_vel='WindVel'           # UPDATE: Wind speed (ASTM only)
)

# Reset filter to include any calculated columns
sim.reset_filter()

# Verify regression columns
sim.regression_cols

### PVsyst Filtering and Regression

Apply the same filtering workflow to PVsyst data, then fit regression.

In [ ]:
# Apply same filtering to PVsyst data
sim.reset_filter()

# Apply filters (use same parameters as measured data)
sim.filter_irr(MIN_IRR, MAX_IRR)
# sim.filter_sensors()  # Usually not needed for PVsyst data
# sim.filter_outliers()  # Optional
sim.filter_clearsky()
# sim.filter_shade()  # Remove shading if present
sim.filter_missing()

# Calculate reporting conditions (use same method as measured data)
sim.rep_cond(irr_bal=True, percent_filter=20)

# Filter around reporting conditions
sim.filter_irr(IRR_LOW, IRR_HIGH, ref_val=sim.rc['poa'][0])

# Fit regression
sim.fit_regression(filter=True, summary=True)
sim.fit_regression(filter=False, summary=True)

print("\nPVsyst filtering summary:")
sim.get_summary()

## 13. Capacity Test Results

**Parameter:** `captest_results(sim, das, nameplate, tolerance, check_pvalues, pval, print_res)` - Calculates capacity ratio and pass/fail

**Recommended:** Use `captest_results_check_pvalues()` to verify coefficient significance

**Reference:** See `docs/user_guide/configurable_parameters.md` section 11 for details.

In [ ]:
# Calculate capacity test results
# UPDATE nameplate and tolerance for your project

# Option 1: Standard results
results = ct.captest_results(
    sim,                      # PVsyst CapData object
    meas,                     # Measured CapData object
    nameplate=AC_NAMEPLATE,   # AC nameplate capacity (Watts)
    tolerance=TOLERANCE,       # Tolerance string (e.g., '+/- 7', '- 5')
    check_pvalues=False,      # Set to True to check coefficient significance
    pval=0.05,                # P-value threshold (default: 0.05)
    print_res=True            # Print results summary
)

# Option 2: Results with p-value checking (recommended)
# results = ct.captest_results_check_pvalues(
#     sim,
#     meas,
#     nameplate=AC_NAMEPLATE,
#     tolerance=TOLERANCE,
#     print_res=True
# )

## 14. Points Required Verification

**Parameter:** `get_pts_required(hrs_req)` - Verifies sufficient data points

**Default:** 12.5 hours = 750 points for 1-minute data (ASTM E2848 requirement)

**Reference:** See `docs/user_guide/configurable_parameters.md` section 12 for details.

In [ ]:
# Verify sufficient data points
# Check if filtered data meets minimum hours requirement
meas.get_pts_required(hrs_req=HRS_REQ)

# Print detailed points summary
meas.print_points_summary(hrs_req=HRS_REQ)

# Also check PVsyst data
sim.get_pts_required(hrs_req=HRS_REQ)
sim.print_points_summary(hrs_req=HRS_REQ)

## 15. Visualization and Data Exploration (Optional)

Use pvcaptest plotting functions to visualize data and filtering results.

**Available methods:**
- `plot()` - Interactive dashboard for timeseries exploration
- `scatter_hv()` - Scatter plots of power vs irradiance
- `scatter_filters()` - Overlay of filtering steps
- `timeseries_filters()` - Timeseries view of filtering

**Reference:** See plotting documentation for details.

In [ ]:
# Example visualizations (uncomment to use)

# Interactive plotting dashboard
# meas.plot(width=1200)

# Scatter plot of power vs irradiance
# meas.scatter_hv(timeseries=True)

# Overlay of filtering steps (scatter plot)
# meas.scatter_filters()

# Overlay of filtering steps (timeseries)
# meas.timeseries_filters().opts(width=1200)

## 16. Export Results (Optional)

Export filtering documentation and results for reporting.

In [ ]:
# Export filtering documentation (uncomment to use)
# Combines filtering table with data for documentation

# filtering_doc = pd.concat([meas.get_filtering_table(), meas.data], axis=1)
# filtering_doc.to_csv('./filtering_documentation_measured.csv')

# Export summary tables
# meas.get_summary().to_csv('./filtering_summary_measured.csv')
# sim.get_summary().to_csv('./filtering_summary_pvsyst.csv')